# MT5 AI Trader — Kaggle Offline Train + Walk-Forward

Notebook dành riêng cho Kaggle.

Luồng offline only:

```text
clone/pull repo -> install deps -> find CSV dataset -> train -> walk-forward -> zip outputs
```

Không chạy MT5, paper, demo, live trên Kaggle.


In [ ]:
# 1) Kaggle environment check
import os, sys, platform, glob
print("Python:", sys.version)
print("Platform:", platform.platform())
print("CWD:", os.getcwd())
print("Kaggle input dirs:", glob.glob("/kaggle/input/*"))
!nproc || true
!df -h /kaggle/working || true


In [ ]:
# 2) Clone or update repo into /kaggle/working
import os

REPO_URL = "https://github.com/quoccuongphan300992-cmd/mt5-ai-trader.git"
REPO_DIR = "/kaggle/working/mt5-ai-trader"

if not os.path.exists(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    %cd "$REPO_DIR"
    !git fetch origin main
    !git checkout main
    !git pull --ff-only origin main

%cd "$REPO_DIR"
!git log -1 --oneline
!ls -lh


In [ ]:
# 3) Install dependencies
import os, pathlib

REPO_DIR = "/kaggle/working/mt5-ai-trader"
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
print("CWD:", os.getcwd())
print("Files:", os.listdir("."))

if os.path.exists("requirements-colab.txt"):
    !pip install -q -r requirements-colab.txt
elif os.path.exists("requirements.txt"):
    !pip install -q -r requirements.txt
else:
    print("requirements file missing; installing fallback packages")
    !pip install -q pandas numpy scikit-learn joblib pytest MetaTrader5 ta


## CSV dataset setup

Kaggle không upload trực tiếp vào repo như Colab.

Cách làm:

1. Tạo Kaggle Dataset chứa file CSV, ví dụ `EURUSD_H1.csv`.
2. Trong notebook, bấm **Add Input**.
3. Chọn dataset đó.
4. Cell dưới tự tìm file CSV trong `/kaggle/input`.

CSV cần cột:

```text
time,open,high,low,close,tick_volume,spread,real_volume
```


In [ ]:
# 4) Locate CSV from Kaggle input and copy into repo data/
import os, glob, shutil

SYMBOL = "EURUSD"
TIMEFRAME = "H1"
BARS = 50000

# Nếu có nhiều CSV, sửa dòng này thành path cụ thể, ví dụ:
# KAGGLE_CSV = "/kaggle/input/your-dataset/EURUSD_H1.csv"
KAGGLE_CSV = None

if KAGGLE_CSV is None:
    candidates = glob.glob("/kaggle/input/**/*.csv", recursive=True)
    print("CSV candidates:")
    for p in candidates:
        print(" -", p)
    if not candidates:
        raise FileNotFoundError("No CSV found under /kaggle/input. Add your CSV dataset via Add Input.")
    preferred = [p for p in candidates if "EURUSD" in os.path.basename(p).upper() and "H1" in os.path.basename(p).upper()]
    KAGGLE_CSV = preferred[0] if preferred else candidates[0]

LOCAL_CSV = "data/EURUSD_H1.csv"
os.makedirs("data", exist_ok=True)
shutil.copy2(KAGGLE_CSV, LOCAL_CSV)

print("Using CSV:", KAGGLE_CSV)
print("Copied to:", os.path.abspath(LOCAL_CSV), os.path.getsize(LOCAL_CSV), "bytes")
!head -5 "$LOCAL_CSV"


In [ ]:
# 5) Train model
!python main.py train --csv "$LOCAL_CSV" --symbol "$SYMBOL" --timeframe "$TIMEFRAME" --bars "$BARS"
!ls -lh models reports


In [ ]:
# 6) Verify upgraded feature set
import json
meta = json.load(open("models/metadata.json"))
required = [
    "adx_14",
    "atr_percentile_100",
    "spread_percentile_100",
    "spread_to_atr",
    "is_london_session",
    "trend_stack_bear",
]
print("feature_count:", meta.get("feature_count"))
print("has_new_features:", all(f in meta.get("features", []) for f in required))
print("missing:", [f for f in required if f not in meta.get("features", [])])


In [ ]:
# 7) SELL walk-forward: no session filter, current best next test
!python main.py walk-forward --csv "$LOCAL_CSV" --symbol "$SYMBOL" --timeframe "$TIMEFRAME" --bars "$BARS" --direction SELL --min 0.52 --max 0.56 --step 0.01 --min-atr-percentile 0.1 --max-atr-percentile 0.95 --max-spread-percentile 0.9


In [ ]:
# 8) Optional: nới spread nếu trade vẫn thiếu
# Chạy cell này nếu cell trên total_trades < 60 nhưng PF/expectancy còn tốt.
# !python main.py walk-forward --csv "$LOCAL_CSV" --symbol "$SYMBOL" --timeframe "$TIMEFRAME" --bars "$BARS" --direction SELL --min 0.52 --max 0.56 --step 0.01 --min-atr-percentile 0.1 --max-atr-percentile 0.95 --max-spread-percentile 0.95


In [ ]:
# 9) Optional: test BUY if SELL still fails
# !python main.py walk-forward --csv "$LOCAL_CSV" --symbol "$SYMBOL" --timeframe "$TIMEFRAME" --bars "$BARS" --direction BUY --min 0.52 --max 0.56 --step 0.01 --min-atr-percentile 0.1 --max-atr-percentile 0.95 --max-spread-percentile 0.9


In [ ]:
# 10) Show walk-forward summary
import pandas as pd, json, os
summary = pd.read_csv("reports/walk_forward_summary.csv")
cols = [
    "threshold",
    "direction",
    "candidate_pass",
    "total_trades",
    "positive_expectancy_folds",
    "positive_pf_folds",
    "overall_profit_factor",
    "overall_expectancy_r",
    "max_fold_drawdown_pct",
    "total_net_profit",
]
display(summary[[c for c in cols if c in summary.columns]])
print(open("reports/walk_forward_summary.json", encoding="utf-8").read())


In [ ]:
# 11) Safety gate
import pandas as pd
summary = pd.read_csv("reports/walk_forward_summary.csv")
passed = summary[summary["candidate_pass"] == True] if "candidate_pass" in summary.columns else pd.DataFrame()

if passed.empty:
    print("candidate_pass=false. Không chạy paper/demo/live.")
else:
    print("Có candidate_pass=true. Vẫn cần review thủ công trước paper/demo/live:")
    display(passed)


In [ ]:
# 12) Zip outputs for Kaggle download
!zip -r /kaggle/working/mt5_ai_outputs_kaggle.zip models reports
!ls -lh /kaggle/working/mt5_ai_outputs_kaggle.zip
